In [7]:
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

user_schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("signup_date", StringType(), nullable=True),
    StructField("plan", StringType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    StructField("marketing_opt_in", BooleanType(), nullable=True),
])

item_schema = StructType([
    StructField("item_id", IntegerType(), nullable=False),
    StructField("category", StringType(), nullable=True),
    StructField("tags", ArrayType(StringType()), nullable=True),
])

events_schema = StructType([
    StructField("ts", StringType(), nullable=True),
    StructField("event", StringType(), nullable=True),
    StructField("user_id", IntegerType(), nullable=True),
    StructField("item_id", IntegerType(), nullable=True),
    StructField("context", StructType([
        StructField("country", StringType(), nullable=True),
        StructField("device", StringType(), nullable=True),
        StructField("locale", StringType(), nullable=True),
        StructField("session_id", StringType(), nullable=True),
    ]), nullable=True),
    StructField("props", StructType([
        StructField("price", DoubleType(), nullable=True),
        StructField("payment_method", StringType(), nullable=True),
        StructField("dwell_ms", IntegerType(), nullable=True),
    ]), nullable=True),
    StructField("exp", StructType([
        StructField("ab_group", StringType(), nullable=True),
    ]), nullable=True),
])


In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("hw2") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 21:31:50 WARN Utils: Your hostname, MacBook-Air-Ivan-4.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.19 instead (on interface en0)
26/03/11 21:31:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 21:32:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
import os

data_dir = "file://" + os.getcwd() + "/data"

users_df = spark.read.json(f"{data_dir}/users.jsonl", schema=user_schema)
items_df = spark.read.json(f"{data_dir}/items.jsonl", schema=item_schema)
events_df = spark.read.json(f"{data_dir}/events/", schema=events_schema)

In [14]:
for name, df in [("users", users_df), ("items", items_df), ("events", events_df)]:
    print(f"{name}: {df.count()} records, {df.rdd.getNumPartitions()} partitions")

users: 20000 records, 1 partitions
items: 5000 records, 1 partitions
events: 200000 records, 4 partitions


In [15]:
from pyspark.sql.functions import to_timestamp, to_date

events_df = events_df.withColumn("timestamp", to_timestamp("ts")).withColumn("date", to_date("ts"))

In [16]:
from pyspark.sql.functions import when, col

events_df = events_df.withColumn(
    "revenue",
    when(col("event") == "purchase", col("props.price")).otherwise(0.0)
)

In [17]:
events_df = events_df.filter(col("revenue") >= 0.0)

In [18]:
from pyspark.sql.functions import broadcast

events_df = events_df \
    .join(broadcast(items_df), on = "item_id", how = "left") \
    .join(broadcast(users_df), on = events_df.user_id == users_df.id, how = "left")

We have these sizes:
```
users: 20000 records, 1 partitions
items: 5000 records, 1 partitions
events: 200000 records, 4 partitions
```

events is much larger than items and users. so we can just copy them with broadcast to workers insted of use shuffle. 

In [21]:
from pyspark.sql.functions import count, sum, countDistinct                                                                                                                                     
                                                                                                                                                                                                  
result_df = events_df.groupBy("date", "country", "category").agg(                                                                                                                               
    count("*").alias("events_total"),                                                                                                                                                           
    count(when(col("event") == "purchase", True)).alias("purchases"),
    sum("revenue").alias("revenue"),
    countDistinct("user_id").alias("unique_users"),
)

In [22]:
from pyspark.sql.window import Window                                                                                                                                                           

w = Window.partitionBy("country", "category").orderBy("date").rowsBetween(-6, 0)

result_df = result_df.withColumn("revenue_7d", sum("revenue").over(w))

In [23]:
result_df.write.partitionBy("date").mode("overwrite").parquet(f"{data_dir}/../out/daily_kpi/")  

In [25]:
one_day = spark.read.parquet(f"{data_dir}/../out/daily_kpi/date=2026-02-09/")
one_day.show(5, truncate=False)

+-------+-----------+------------+---------+------------------+------------+------------------+
|country|category   |events_total|purchases|revenue           |unique_users|revenue_7d        |
+-------+-----------+------------+---------+------------------+------------+------------------+
|BR     |books      |80          |2        |414.28999999999996|78          |414.28999999999996|
|BR     |electronics|67          |1        |98.07             |67          |98.07             |
|BR     |fashion    |73          |4        |342.80999999999995|71          |342.80999999999995|
|BR     |home       |85          |2        |177.74            |85          |177.74            |
|BR     |sports     |70          |3        |450.68            |69          |450.68            |
+-------+-----------+------------+---------+------------------+------------+------------------+
only showing top 5 rows


In [29]:
import time

def measure(label, fn):
    start = time.time()
    fn()
    elapsed = time.time() - start
    print(f"{label}: {elapsed:.2f}s")

raw_events = spark.read.json(f"{data_dir}/events/", schema=events_schema) \
    .withColumn("timestamp", to_timestamp("ts")) \
    .withColumn("date", to_date("ts")) \
    .withColumn("revenue", when(col("event") == "purchase", col("props.price")).otherwise(0.0)) \
    .filter(col("revenue") >= 0.0)

measure("Join (no repartition)", lambda: raw_events
    .join(broadcast(items_df), on="item_id", how="left")
    .join(broadcast(users_df), raw_events["user_id"] == users_df["id"], how="left")
    .count())

measure("Join (repartition by user_id)", lambda: raw_events
    .repartition("user_id")
    .join(broadcast(items_df), on="item_id", how="left")
    .join(broadcast(users_df), raw_events["user_id"] == users_df["id"], how="left")
    .count())

measure("Join (repartition round-robin 8)", lambda: raw_events
    .repartition(8)
    .join(broadcast(items_df), on="item_id", how="left")
    .join(broadcast(users_df), raw_events["user_id"] == users_df["id"], how="left")
    .count())

Join (no repartition): 0.45s
Join (repartition by user_id): 0.35s
Join (repartition round-robin 8): 0.23s


In [30]:
joined_df = raw_events \
    .join(broadcast(items_df), on="item_id", how="left") \
    .join(broadcast(users_df), raw_events["user_id"] == users_df["id"], how="left")

agg_expr = [
    count("*").alias("events_total"),
    count(when(col("event") == "purchase", True)).alias("purchases"),
    sum("revenue").alias("revenue"),
    countDistinct("user_id").alias("unique_users"),
]

measure("Agg (no repartition)", lambda: joined_df
    .groupBy("date", "country", "category").agg(*agg_expr)
    .count())

measure("Agg (repartition by date/country/category)", lambda: joined_df
    .repartition("date", "country", "category")
    .groupBy("date", "country", "category").agg(*agg_expr)
    .count())

Agg (no repartition): 0.74s
Agg (repartition by date/country/category): 0.57s


In [31]:
out_path = f"{data_dir}/../out/daily_kpi_bench/"


measure("Write (no repartition)", lambda: result_df
    .write.partitionBy("date").mode("overwrite").parquet(out_path + "baseline"))

measure("Write (coalesce 4)", lambda: result_df
    .coalesce(4)
    .write.partitionBy("date").mode("overwrite").parquet(out_path + "coalesce4"))

measure("Write (repartition by date)", lambda: result_df
    .repartition("date")
    .write.partitionBy("date").mode("overwrite").parquet(out_path + "repart_date"))

Write (no repartition): 2.24s


Write (coalesce 4): 2.11s


Write (repartition by date): 2.09s


### 1. Before Join
| Strategy | Time |
|---|---|
| No repartition (4 partitions) | 0.45s |
| Repartition by user_id | 0.35s |
| Round-robin repartition(8) | **0.23s** |

Round-robin to 8 partitions was the fastest (~2x improvement). Since we use `broadcast()` joins, the shuffle cost of repartitioning is minimal, but doubling the partition count (4 -> 8) gives better parallelism across CPU cores. Repartitioning by `user_id` helped slightly but the hash distribution adds overhead that isn't needed for broadcast joins.

### 2. Before Aggregation
| Strategy | Time |
|---|---|
| No repartition | 0.74s |
| Repartition by date/country/category | **0.57s** |

Pre-partitioning by the groupBy keys reduced time by ~23%. This is because data with the same keys ends up on the same partition, so the aggregation stage needs less shuffling.

### 3. Before Write
| Strategy | Time |
|---|---|
| No repartition | 2.24s |
| Coalesce(4) | 2.11s |
| Repartition by date | **2.09s** |

Write times are very similar (~2.1-2.2s). The dataset is small enough that write I/O dominates rather than shuffle cost. `coalesce(4)` and `repartition("date")` both marginally help by reducing the number of output files per partition folder, but the gains are negligible at this scale.